In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [2]:
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD_2"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [3]:
fecha_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=0", con=connection2)
fecha_ini= fecha_ini.iloc[0, 0]

fecha_fin = pd.read_sql_query("SELECT fec_ter FROM etl_act where id_mod=0", con=connection2)
fecha_fin= fecha_fin.iloc[0, 0]

In [4]:
dias_por_intervalo = 15

# Inicializa la fecha actual
fecha_actual = fecha_ini

while fecha_actual <= fecha_fin:
	inicioTiempo = time.time()
	now_inicio = datetime.now()

	fecha_ini_intervalo = fecha_actual
	fecha_fin_intervalo = fecha_actual + timedelta(days=dias_por_intervalo - 1)

	if fecha_fin_intervalo > fecha_fin:
		fecha_fin_intervalo = fecha_fin

	fecha_ini_str = fecha_ini_intervalo.strftime('%Y-%m-%d')
	fecha_fin_str = (fecha_fin_intervalo + timedelta(days=1)).strftime('%Y-%m-%d')
	fecha_fin_display_str = fecha_fin_intervalo.strftime('%Y-%m-%d')

	print(f"Procesando de {fecha_ini_str} al {fecha_fin_display_str}")

	now1 = datetime.now()
	now2 = datetime.now().strftime('%Y-%m-%d')

	query=f"UPDATE etl_act SET fec_act ='{now1}' WHERE id_mod=0"

	c1= text(query)
	connection2.execute(c1)

	tabla='ctsci10'
	col_tabla = "solcitafec"
	dat= "cext00_essi"
	col_dat='fec_sol'


	oracledb.init_oracle_client()
	oracledb.version = "8.3.0"
	sys.modules["cx_Oracle"] = oracledb
	un = config("USER4_BDI_POSTGRES")
	pw = config("PASS4_BDI_POSTGRES")
	hostname=config("HOST4_BDI_POSTGRES")
	service_name="WNET"
	port = 1527

	engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
			"host": hostname,
			"port": port,
			"service_name": service_name
		}
	)

	connection0 = engine0.connect()


	query1 = f"""
	SELECT
		c.SOLCITANUM,
		c.SOLCITAFEC,
		c.SOLCITAORICENASICOD,
		c.SOLCITACENASICOD,
		c.SOLCITAPACSECNUM,
		c.SOLCITAAREHOSCOD,
		c.SOLCITASERVHOSCOD,
		c.SOLCITAACTCOD,
		c.SOLCITAACTESPCOD,
		c.SOLCITATIPOCITACOD,
		c.SOLCITAFECPREF,
		c.DIAPREFSOLCITCOD,
		c.TURPREFSOLCITACOD,
		c.ESTATENSOLCITACOD,
		c.SOLCITAORICENASIORICOD,
		c.SOLCITACENASIORICOD,
		c.SOLCITAACTMEDORINUM,
		c.SOLCITAORICENASICITCOD,
		c.SOLCITACENASICITCOD,
		c.SOLCITAACTMEDCITNUM,
		c.SOLCITAESTREGCOD,
		c.SOLCITACREFEC,
		c.SOLCITAMODFEC,
		c.SOLCITAFECANU,
		c.SOLCITASECNUM,
		c.SOLCITACPSCOD
	FROM {tabla} c
	WHERE c.{col_tabla} >= TO_DATE('{fecha_ini_str}', 'YYYY-MM-DD') AND c.{col_tabla} < TO_DATE('{fecha_fin_str}', 'YYYY-MM-DD')
	"""

	base1 = pd.read_sql_query(query1, con=connection0)
	connection0.close()
	engine0.dispose()

	base1 = base1.rename(columns={
		'solcitanum': 'num_sol',
		'solcitafec': 'fec_sol',
		'solcitaoricenasicod': 'cod_ori',
		'solcitacenasicod': 'cod_cas',
		'solcitapacsecnum': 'pac_sec',
		'solcitaarehoscod': 'cod_are',
		'solcitaservhoscod': 'cod_ser',
		'solcitaactcod': 'cod_act',
		'solcitaactespcod': 'cod_sub',
		'solcitatipocitacod': 'cod_tci',
		'solcitafecpref': 'fec_pref',
		'diaprefsolcitcod': 'cod_diapref',
		'turprefsolcitacod': 'cod_turpref',
		'estatensolcitacod': 'cod_estsolcit',
		'solcitaoricenasioricod': 'cod_ori_sol',
		'solcitacenasioricod': 'cod_cas_sol',
		'solcitaactmedorinum': 'act_med_sol',
		'solcitaoricenasicitcod': 'cod_ori_gen',
		'solcitacenasicitcod': 'cod_cas_gen',
		'solcitaactmedcitnum': 'act_med_gen',
		'solcitaestregcod': 'est_reg',
		'solcitacrefec': 'fec_cre',
		'solcitamodfec': 'fec_mod',
		'solcitafecanu': 'fec_anu',
		'solcitasecnum': 'sec_num',
		'solcitacpscod': 'cod_cps'
	})

	base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

	oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
	oricas['ori_cod'] = oricas['ori_cod'].str.strip()
	base1['cod_ori'] = base1['cod_ori'].str.strip()
	oricas=oricas.rename(columns={"ori_cod":"cod_ori"})
	base1=pd.merge(base1,oricas,how='left',on='cod_ori')

	base1=base1.drop('cod_ori',axis=1)

	oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
	oricas['ori_cod'] = oricas['ori_cod'].str.strip()
	base1['cod_ori_sol'] = base1['cod_ori_sol'].str.strip()
	oricas=oricas.rename(columns={"id_oricas":"id_ori_sol"})
	oricas=oricas.rename(columns={"ori_cod":"cod_ori_sol"})
	base1=pd.merge(base1,oricas,how='left',on='cod_ori_sol')

	base1=base1.drop('cod_ori_sol',axis=1)

	oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
	oricas['ori_cod'] = oricas['ori_cod'].str.strip()
	base1['cod_ori_gen'] = base1['cod_ori_gen'].str.strip()
	oricas=oricas.rename(columns={"id_oricas":"id_ori_gen"})
	oricas=oricas.rename(columns={"ori_cod":"cod_ori_gen"})
	base1=pd.merge(base1,oricas,how='left',on='cod_ori_gen')

	base1=base1.drop('cod_ori_gen',axis=1)
	
	cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM dim_cas where cod_red is not null", con=connection2)
	cas = cas.dropna()
	cas['cod_cas'] = cas['cod_cas'].str.strip()
	base1['cod_cas'] = base1['cod_cas'].str.strip()
	base1=pd.merge(base1,cas,how='left',on='cod_cas')

	base1=base1.drop('cod_cas',axis=1)
	
	red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
	red['cod_red'] = red['cod_red'].str.strip()
	base1['cod_red'] = base1['cod_red'].str.strip()
	base1=pd.merge(base1,red,how='left',on='cod_red')

	base1=base1.drop('cod_red',axis=1)

	cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM dim_cas where cod_red is not null", con=connection2)
	cas = cas.dropna()
	cas['cod_cas'] = cas['cod_cas'].str.strip()
	cas=cas.rename(columns={"id_cas":"id_cas_sol"})
	cas=cas.rename(columns={"cod_cas":"cod_cas_sol"})
	cas=cas.rename(columns={"cod_red":"cod_red_sol"})
	base1['cod_cas_sol'] = base1['cod_cas_sol'].str.strip()
	base1=pd.merge(base1,cas,how='left',on='cod_cas_sol')

	base1=base1.drop('cod_cas_sol',axis=1)
	
	red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
	red['cod_red'] = red['cod_red'].str.strip()
	red=red.rename(columns={"id_red":"id_red_sol"})
	red=red.rename(columns={"cod_red":"cod_red_sol"})
	base1['cod_red_sol'] = base1['cod_red_sol'].str.strip()
	base1=pd.merge(base1,red,how='left',on='cod_red_sol')

	base1=base1.drop('cod_red_sol',axis=1)

	cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM dim_cas where cod_red is not null", con=connection2)
	cas = cas.dropna()
	cas['cod_cas'] = cas['cod_cas'].str.strip()
	cas=cas.rename(columns={"id_cas":"id_cas_gen"})
	cas=cas.rename(columns={"cod_cas":"cod_cas_gen"})
	cas=cas.rename(columns={"cod_red":"cod_red_gen"})
	base1['cod_cas_gen'] = base1['cod_cas_gen'].str.strip()
	base1=pd.merge(base1,cas,how='left',on='cod_cas_gen')

	base1=base1.drop('cod_cas_gen',axis=1)
	
	red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
	red['cod_red'] = red['cod_red'].str.strip()
	red=red.rename(columns={"id_red":"id_red_gen"})
	red=red.rename(columns={"cod_red":"cod_red_gen"})
	base1['cod_red_gen'] = base1['cod_red_gen'].str.strip()
	base1=pd.merge(base1,red,how='left',on='cod_red_gen')

	base1=base1.drop('cod_red_gen',axis=1)

	are = pd.read_sql_query(f"SELECT id_area,cod_are FROM dim_areas", con=connection2)
	are['cod_are'] = are['cod_are'].str.strip()
	base1['cod_are'] = base1['cod_are'].str.strip()
	base1=pd.merge(base1,are,how='left',on='cod_are')

	base1=base1.drop('cod_are',axis=1)

	serv= pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
	serv['cod_ser'] = serv['cod_ser'].str.strip()
	base1['cod_ser'] = base1['cod_ser'].str.strip()
	base1=pd.merge(base1,serv,how='left',on='cod_ser')

	base1=base1.drop('cod_ser',axis=1)
	
	activi = pd.read_sql_query(f"SELECT id_activi,cod_act FROM dim_activi", con=connection2)
	activi['cod_act'] = activi['cod_act'].str.strip()
	base1['cod_act'] = base1['cod_act'].str.strip()
	activi=activi.rename(columns={"id_activi":"id_acti"})
	base1=pd.merge(base1,activi,how='left',on='cod_act')
	
	subacti = pd.read_sql_query(f"SELECT id_subacti,cod_sub,cod_act FROM dim_subacti", con=connection2)
	subacti['cod_act'] = subacti['cod_act'].str.strip()
	subacti['cod_sub'] = subacti['cod_sub'].str.strip()
	base1['cod_act'] = base1['cod_act'].str.strip()
	base1['cod_sub'] = base1['cod_sub'].str.strip()
	subacti["KEY"]=subacti["cod_sub"]+subacti["cod_act"]
	subacti=subacti.drop(["cod_sub",'cod_act'], axis=1)
	base1["KEY"]=base1["cod_sub"].astype(str)+base1['cod_act'].astype(str)
	base1["KEY"]=base1["KEY"].str.replace(' ', '', regex=True)
	subacti["KEY"]=subacti["KEY"].str.replace(' ', '', regex=True)
	base1 = pd.merge(base1,subacti,how='left',on='KEY')

	base1=base1.drop('KEY', axis=1)
	base1=base1.drop('cod_act',axis=1)
	base1=base1.drop('cod_sub',axis=1)

	tipcit = pd.read_sql_query(f"SELECT id_tipcit,cod_tci FROM dim_tipcit", con=connection2)
	tipcit['cod_tci'] = tipcit['cod_tci'].str.strip()
	base1['cod_tci'] = base1['cod_tci'].str.strip()
	base1=pd.merge(base1,tipcit,how='left',on='cod_tci')

	base1=base1.drop('cod_tci', axis=1)

	diapref = pd.read_sql_query(f"SELECT id_diaprefcit,cod_diapref FROM dim_diaprefcit", con=connection2)
	diapref['cod_diapref'] = diapref['cod_diapref'].str.strip()
	base1['cod_diapref'] = base1['cod_diapref'].str.strip()
	base1=pd.merge(base1,diapref,how='left',on='cod_diapref')

	base1=base1.drop('cod_diapref', axis=1)

	turpref = pd.read_sql_query(f"SELECT id_turprefcit,cod_turpref FROM dim_turnoprefcit", con=connection2)
	turpref['cod_turpref'] = turpref['cod_turpref'].str.strip()
	base1['cod_turpref'] = base1['cod_turpref'].str.strip()
	base1=pd.merge(base1,turpref,how='left',on='cod_turpref')

	base1=base1.drop('cod_turpref', axis=1)

	estsolcit = pd.read_sql_query(f"SELECT id_estsolcit,cod_estsolcit FROM dim_estsolcit", con=connection2)
	estsolcit['cod_estsolcit'] = estsolcit['cod_estsolcit'].str.strip()
	base1['cod_estsolcit'] = base1['cod_estsolcit'].str.strip()
	base1=pd.merge(base1,estsolcit,how='left',on='cod_estsolcit')

	base1=base1.drop('cod_estsolcit', axis=1)

	estreg = pd.read_sql_query(f"SELECT id_estreg,cod_est FROM dim_estreg", con=connection2)
	estreg=estreg.rename(columns={"cod_est":"est_reg"})
	estreg['est_reg'] = estreg['est_reg'].str.strip()
	base1['est_reg'] = base1['est_reg'].str.strip()
	base1=pd.merge(base1,estreg,how='left',on='est_reg')

	base1=base1.drop('est_reg',axis=1)
	
	cps = pd.read_sql_query(f"SELECT id_cps,cod_cps FROM dim_cps", con=connection2)
	cps['cod_cps'] = cps['cod_cps'].str.strip()
	base1['cod_cps'] = base1['cod_cps'].str.strip()
	base1=pd.merge(base1,cps,how='left',on='cod_cps')

	base1=base1.drop('cod_cps',axis=1)

	df1_columns = set(base1.columns)
	df2_columns = set(base2.columns) 
	different_columns = df2_columns - df1_columns
	different_columns

	borrando = f"DELETE FROM {dat} WHERE {col_dat} >='{fecha_ini_str}' and {col_dat} <'{fecha_fin_str}'"
	borrado = connection2.execute(borrando)

	comunes = set(base1.columns).intersection(set(base2.columns)) 
	base1 = base1[list(comunes)]
	base1.to_sql(name=f'{dat}', con=engine2, if_exists='append', index=False,chunksize=500000)

	fecha_actual = fecha_fin_intervalo + timedelta(days=1)

	finproceso = time.time()
	tiempoproceso = finproceso - inicioTiempo
	tiempoproceso = round(tiempoproceso, 3)
	print("Proceso completado satisfactoriamente en " + str(tiempoproceso) + " segundos")

query2=f"UPDATE etl_act SET fec_ini ='{now2}' WHERE id_mod=0"
c2= text(query2)
connection2.execute(c2)


Procesando de 2024-07-08 al 2024-07-22
Proceso completado satisfactoriamente en 252.727 segundos
Procesando de 2024-07-23 al 2024-08-06
Proceso completado satisfactoriamente en 12.677 segundos
Procesando de 2024-08-07 al 2024-08-21
Proceso completado satisfactoriamente en 0.798 segundos
Procesando de 2024-08-22 al 2024-09-05
Proceso completado satisfactoriamente en 0.791 segundos
Procesando de 2024-09-06 al 2024-09-20
Proceso completado satisfactoriamente en 0.848 segundos
Procesando de 2024-09-21 al 2024-10-05
Proceso completado satisfactoriamente en 0.652 segundos
Procesando de 2024-10-06 al 2024-10-20
Proceso completado satisfactoriamente en 1.386 segundos
Procesando de 2024-10-21 al 2024-11-04
Proceso completado satisfactoriamente en 0.862 segundos
Procesando de 2024-11-05 al 2024-11-19
Proceso completado satisfactoriamente en 0.768 segundos
Procesando de 2024-11-20 al 2024-12-04
Proceso completado satisfactoriamente en 0.695 segundos
Procesando de 2024-12-05 al 2024-12-19
Proceso 

In [5]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()